## Imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
""" Load SP500 data """

sp_500_df = pd.read_csv(
    "../../data/raw/sp500_2010_historical.csv",
    index_col=0,      # first column is the date
    parse_dates=True  # parse dates automatically
)

# Check it loaded correctly
print(sp_500_df.head())
print(sp_500_df.tail())
print(sp_500_df.shape)

                  Close         High          Low         Open      Volume
Date                                                                      
2010-01-04  1132.989990  1133.869995  1116.560059  1116.560059  3991400000
2010-01-05  1136.520020  1136.630005  1129.660034  1132.660034  2491020000
2010-01-06  1137.140015  1139.189941  1133.949951  1135.709961  4972660000
2010-01-07  1141.689941  1142.459961  1131.319946  1136.270020  5270680000
2010-01-08  1144.979980  1145.390015  1136.219971  1140.520020  4389590000
                  Close         High          Low         Open      Volume
Date                                                                      
2026-06-12  7431.459961  7456.399902  7363.009766  7410.850098  4950530000
2026-06-15  7554.290039  7577.919922  7516.750000  7516.750000  5674670000
2026-06-16  7511.350098  7564.959961  7508.680176  7548.779785  5286210000
2026-06-17  7420.100098  7532.169922  7402.609863  7524.500000  5883740000
2026-06-18  7500.580078  

In [3]:
""" Load market data """

market_df = pd.read_csv(
    "../../data/raw/fixed_market_data.csv",
    index_col=0, 
    sep=';'     
)

# Check it loaded correctly
print(market_df.head())
print(market_df.tail())
print(market_df.shape)

          EMP    PE    CAPE    DY     Rho   MOV     IR      RR    Y02    Y10  \
Date                                                                           
4/10/88  2,0%  12,9  14,469   3,6  -0,083    118  6,02  -1,061  7,459  8,484   
4/17/88  2,0%  12,4   13,96  3,75  -0,078    121  5,88   -0,76  7,582  8,737   
4/24/88  2,0%  12,4   13,95  3,75  -0,051    123  5,83   -0,76  7,618  8,773   
5/1/88   2,0%  12,5  14,036  3,75  -0,054  124,2  5,98   -0,76  7,728  8,873   
5/8/88   2,0%  12,3  13,761  3,82  -0,079  118,4  6,29   -0,76  7,885   8,99   

         ...       YSS          NYF     _AU   _DXY    _LCP     _TY   _OIL  \
Date     ...                                                                
4/10/88  ...  0,251622  6,896895065   450,5  89,14  2473,7   97,99  16,87   
4/17/88  ...  0,328976  2,631748517  456,25  88,31  2328,2  96,534  18,32   
4/24/88  ...  0,156475  2,631748517  449,25  88,89  2201,4   96,47  18,13   
5/1/88   ...  0,146747  2,631748517     449  89,16  21

## Feature Engineering

In [4]:
# Log returns to normalize data
level_cols = ['Close', 'High', 'Low', 'Open']
for col in level_cols:
        if col in sp_500_df.columns:
            # ln(P_t / P_t-1)
            sp_500_df[f'{col}_LogRet'] = np.log(sp_500_df[col] / sp_500_df[col].shift(1))

sp_500_df.dropna(inplace=True)

sp_500_df.head()

,Close,High,Low,Open,Volume,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet
Date,,,,,,,,,
2010-01-05,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,0.003111,0.002431,0.011664,0.014316
2010-01-06,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,0.000545,0.002250,0.003790,0.002689
2010-01-07,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,0.003993,0.002866,-0.002322,0.000493
2010-01-08,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,0.002878,0.002561,0.004322,0.003733
2010-01-11,1146.979980,1149.739990,1142.020020,1145.959961,4255780000,0.001745,0.003791,0.005092,0.004758


In [5]:
# 1. Automatically find all columns that are currently 'object' (text)
object_columns = market_df.select_dtypes(include=['object']).columns

# 2. Loop through only the messy columns and fix them
for col in object_columns:
    # Replace European commas with dots and strip invisible spaces
    market_df[col] = market_df[col].astype(str).str.replace(',', '.').str.strip()
    
    # Force conversion to float64. Any stubborn text (like "N/A" or "-") becomes NaN
    market_df[col] = pd.to_numeric(market_df[col], errors='coerce')

# 3. Print the data types again to verify total victory
print(market_df.dtypes)

EMP     float64
PE      float64
CAPE    float64
DY      float64
Rho     float64
MOV     float64
IR      float64
RR      float64
Y02     float64
Y10     float64
STP     float64
CF      float64
MG      float64
RV      float64
ED      float64
UN      float64
GDP     float64
M2      float64
CPI     float64
DIL     float64
YSS     float64
NYF     float64
_AU     float64
_DXY    float64
_LCP    float64
_TY     float64
_OIL    float64
_MKT    float64
_VA     float64
_GR     float64
dtype: object


In [15]:
import pandas as pd
import numpy as np


# Drop the known flatlined columns right away to keep the dataset lean
market_df = market_df.drop(columns=['EMP', 'MG'], errors='ignore')

# Automatically find and fix the Datastream string/comma issue for all remaining columns
object_columns = market_df.select_dtypes(include=['object']).columns
for col in object_columns:
    market_df[col] = market_df[col].astype(str).str.replace(',', '.').str.strip()
    market_df[col] = pd.to_numeric(market_df[col], errors='coerce')

# ---------------------------------------------------------
# 2. Base Stationarity (Log Returns for Assets)
# ---------------------------------------------------------
asset_cols = ['_AU', '_DXY', '_LCP', '_TY', '_OIL', '_MKT', '_VA', '_GR']
for col in asset_cols:
    if col in market_df.columns:
        market_df[f'{col}_LogRet'] = np.log(market_df[col] / market_df[col].shift(1))

# ---------------------------------------------------------
# 3. CAPE & USD (Currency Squeeze Features) - WEEKLY
# ---------------------------------------------------------
if 'CAPE' in market_df.columns and '_DXY' in market_df.columns:
    market_df['CAPE_Yield_Pct'] = (1 / market_df['CAPE']) * 100
    
    # 3 Months = 13 Weeks
    market_df['USD_Momentum_3M'] = market_df['_DXY'].pct_change(13)
    market_df['Valuation_Currency_Risk'] = market_df['CAPE'] * market_df['USD_Momentum_3M']

# ---------------------------------------------------------
# 4. Cross-Asset Ratios & Spreads - WEEKLY
# ---------------------------------------------------------
# Copper/Gold Ratio (Growth proxy)
if '_LCP' in market_df.columns and '_AU' in market_df.columns:
    market_df['Copper_Gold_Ratio'] = market_df['_LCP'] / market_df['_AU']
    market_df['Copper_Gold_Trend_3M'] = market_df['Copper_Gold_Ratio'].pct_change(13) 
    
# Equity Risk Premium (ERP)
if 'PE' in market_df.columns and 'Y10' in market_df.columns:
    market_df['Earnings_Yield_Pct'] = (1 / market_df['PE']) * 100
    market_df['ERP'] = market_df['Earnings_Yield_Pct'] - market_df['Y10']

# ---------------------------------------------------------
# 5. Macro Acceleration (3-Month Differences) - WEEKLY
# ---------------------------------------------------------
# EMP and MG have been removed from this list
macro_yoy_cols = ['CPI', 'UN', 'GDP', 'M2']
for col in macro_yoy_cols:
    if col in market_df.columns:
        market_df[f'{col}_Acceleration_3M'] = market_df[col] - market_df[col].shift(13)

# ---------------------------------------------------------
# 6. Financial Stress Regimes (Rolling Z-Scores) - WEEKLY
# ---------------------------------------------------------
risk_cols = ['MOV', 'YSS', 'NYF']
for col in risk_cols:
    if col in market_df.columns:
        # 1 Year = 52 Weeks
        rolling_mean = market_df[col].rolling(window=52).mean()
        rolling_std = market_df[col].rolling(window=52).std()
        market_df[f'{col}_ZScore_1Y'] = (market_df[col] - rolling_mean) / rolling_std

# ---------------------------------------------------------
# 7. Clean Up & Finalize
# ---------------------------------------------------------
# Drop the 1988 "warm-up" period rows with NaNs
market_df = market_df.dropna()



In [16]:
# ====================================================================
# Quarterly feature (13-week momentum)
# ====================================================================

# 1. GROWTH: Add Earnings Acceleration 
# (You already have GDP, UN, and Cu/Au ratio)
if 'ED' in market_df.columns:
    market_df['Earnings_Accel_3M'] = market_df['ED'] - market_df['ED'].shift(13)

# 2. INFLATION: Implied Breakeven and Oil Shock
# Implied Inflation Proxy = Nominal Rate (IR) - Real Rate (RR)
if 'IR' in market_df.columns and 'RR' in market_df.columns:
    market_df['Implied_Inflation_Proxy'] = market_df['IR'] - market_df['RR']
# Oil Shock (13-week momentum)
if '_OIL' in market_df.columns:
    market_df['Oil_Shock_3M'] = market_df['_OIL'].pct_change(13)

# 3. INTEREST RATES: Policy Trajectory and Yield Curve
# Fed Policy Trajectory (Rate hike/cut momentum)
if 'IR' in market_df.columns:
    market_df['3M_Interest_Rate_Change'] = market_df['IR'] - market_df['IR'].shift(13)
# Steepening / Flattening Trend
if 'STP' in market_df.columns:
    market_df['Steepness_Trend_3M'] = market_df['STP'] - market_df['STP'].shift(13)

# Drop any new NaNs generated by these shifts
market_df = market_df.dropna()

In [17]:
display(market_df)

,PE,CAPE,DY,Rho,MOV,IR,RR,Y02,Y10,STP,...,_TY_LogRet,_OIL_LogRet,_MKT_LogRet,_VA_LogRet,_GR_LogRet,Earnings_Accel_3M,Implied_Inflation_Proxy,Oil_Shock_3M,3M_Interest_Rate_Change,Steepness_Trend_3M
Date,,,,,,,,,,,,,,,,,,,,,
7/2/89,13.0,15.995,3.43,0.600,119.6,7.990,1.998,8.107,8.076,-0.031,...,0.011481,0.029002,-0.029140,-0.001063,-0.006403,0.0,5.992,0.000987,-0.910,0.441
7/9/89,13.3,16.296,3.36,0.588,117.6,7.730,1.998,7.850,8.012,0.162,...,0.005638,0.023381,0.021906,0.000000,0.000000,0.0,5.732,0.037463,-1.070,0.525
7/16/89,13.6,16.643,3.30,0.600,115.2,7.840,1.804,7.822,8.033,0.211,...,0.000152,-0.020429,0.021914,0.000000,0.000000,0.0,6.036,-0.015957,-0.770,0.495
7/23/89,13.5,16.879,3.26,0.587,109.9,8.120,1.804,7.941,8.059,0.118,...,-0.000125,-0.024373,0.014956,0.000000,0.000000,0.0,6.316,-0.150556,-0.520,0.422
7/30/89,13.8,17.203,3.20,0.555,106.5,7.860,1.804,7.656,7.854,0.198,...,0.014687,-0.100561,0.019889,0.000000,0.000000,0.0,6.056,-0.118744,-0.550,0.429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3/22/26,26.9,35.017,1.22,0.052,108.8,3.708,0.578,3.946,4.381,0.435,...,-0.007439,0.002333,-0.019177,-0.014066,-0.023649,0.0,3.130,0.737852,0.093,-0.194
3/29/26,26.4,34.291,1.25,0.094,111.9,3.698,0.578,3.945,4.432,0.487,...,-0.003108,0.025505,-0.019712,-0.012473,-0.030978,0.0,3.120,0.789046,0.068,-0.129
4/5/26,27.2,35.410,1.21,0.166,81.8,3.698,0.578,3.844,4.345,0.501,...,0.008174,0.111730,0.033367,0.026386,0.041609,0.0,3.120,0.979199,0.073,-0.213


## Aggregation/Windowing

In [7]:
# ---------------------------------------------------------
# 1. Price Action & Volatility Features
# ---------------------------------------------------------
# Daily trading range (proxy for intraday volatility)
sp_500_df['Intraday_Range'] = sp_500_df['High_LogRet'] - sp_500_df['Low_LogRet']
    
# Intraday direction (Close relative to Open)
sp_500_df['Intraday_Return'] = sp_500_df['Close_LogRet'] - sp_500_df['Open_LogRet']

# Rolling 5-day and 21-day realized volatility
sp_500_df['Vol_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).std()
sp_500_df['Vol_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).std()
    
# Cumulative returns over the last week and month
sp_500_df['Return_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).sum()
sp_500_df['Return_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).sum()

# ---------------------------------------------------------
# 2. Volume Features
# ---------------------------------------------------------
# Volume day-over-day percentage change
sp_500_df['Volume_Pct_Change'] = sp_500_df['Volume'].pct_change()
    
# Volume relative to its 21-day moving average (Volume Surge)
vol_ma_21 = sp_500_df['Volume'].rolling(window=21).mean()
sp_500_df['Volume_Surge_Ratio'] = sp_500_df['Volume'] / vol_ma_21

# ---------------------------------------------------------
# 3. Time Series Lags (Memory)
# ---------------------------------------------------------
# Lags of the close return (yesterday, 2 days ago, 3 days ago)
for i in [1, 2, 3]:
    sp_500_df[f'Close_LogRet_Lag{i}'] = sp_500_df['Close_LogRet'].shift(i)
    sp_500_df[f'Volume_Pct_Change_Lag{i}'] = sp_500_df['Volume_Pct_Change'].shift(i)

# ---------------------------------------------------------
# 4. Clean Up
# ---------------------------------------------------------
# Drop rows with NaNs created by rolling windows and lags
sp_500_df = sp_500_df.dropna()
sp_500_df.head()


,Close,High,Low,Open,Volume,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet,Intraday_Range,...,Return_5d,Return_21d,Volume_Pct_Change,Volume_Surge_Ratio,Close_LogRet_Lag1,Volume_Pct_Change_Lag1,Close_LogRet_Lag2,Volume_Pct_Change_Lag2,Close_LogRet_Lag3,Volume_Pct_Change_Lag3
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-03,1097.280029,1102.719971,1093.969971,1100.670044,4285450000,-0.005489,-0.001821,0.005509,0.009696,-0.007330,...,-0.000200,-0.032026,-0.097713,0.899326,0.012890,0.164785,0.014165,-0.246680,-0.009878,-0.007254
2010-02-04,1063.109985,1097.250000,1062.780029,1097.250000,5859690000,-0.031636,-0.004973,-0.028925,-0.003112,0.023952,...,-0.019948,-0.066772,0.367345,1.189642,-0.005489,-0.097713,0.012890,0.164785,0.014165,-0.246680
2010-02-05,1066.189941,1067.130005,1044.500000,1064.119995,6438900000,0.002893,-0.027834,-0.017350,-0.030659,-0.010484,...,-0.007177,-0.064425,0.098847,1.288962,-0.031636,0.367345,-0.005489,-0.097713,0.012890,0.164785
2010-02-08,1056.739990,1071.199951,1056.510010,1065.510010,4089820000,-0.008903,0.003807,0.011433,0.001305,-0.007626,...,-0.030246,-0.077321,-0.364826,0.828036,0.002893,0.098847,-0.031636,0.367345,-0.005489,-0.097713
2010-02-09,1070.520020,1079.280029,1060.060059,1060.060059,5114260000,0.012956,0.007515,0.003355,-0.005128,0.004160,...,-0.030179,-0.067243,0.250485,1.028263,-0.008903,-0.364826,0.002893,0.098847,-0.031636,0.367345


## Model Feature Importance